# 04 · Heston known-model check

Owner: simulation lead.

Simulate a Heston world where the true minimum-variance correction is known, train the residual network on it, and compare the learned correction to the truth.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import load, build_dataset, models, evaluate
from src import heston

In [ ]:
from src.schema import KEY
P = heston.HestonParams()
sim = heston.simulate_chain(P, n_days=1305, seed=0)
sim = sim.drop_duplicates(KEY + ["date"])
sim_pairs = build_dataset.pair_frame(sim, cal=sim["date"].unique(), splits="proposal")
sim_pairs["v_true"] = sim_pairs["date"].map(sim.groupby("date")["v_true"].first())
sim_pairs = sim_pairs[
    (sim_pairs["dte"] >= 14)
    & sim_pairs["delta"].abs().between(0.05, 0.95)
    & (sim_pairs["v_true"] >= 0.005)
].copy()
n = 5000
sim_pairs["true_corr"] = np.concatenate([
    heston.true_mv_correction(sim_pairs.iloc[i:i + n], P)
    for i in range(0, len(sim_pairs), n)
])
sim_pairs = sim_pairs.dropna(subset=["true_corr"])
sim_pairs.groupby("split", observed=True).size()

In [ ]:
feats = ["T", "delta", "moneyness", "iv", "vega", "gamma"]   # theta not simulated
tr, va = sim_pairs[sim_pairs.split == "train"], sim_pairs[sim_pairs.split == "val"]
nn = models.NeuralHedger(residual=True, features=feats).fit(tr, val=va)
sim_pairs["delta_nn_resid"] = nn.predict(sim_pairs)
sim_pairs["delta_true"] = sim_pairs["delta"] + sim_pairs["true_corr"]
evaluate.gain_table(sim_pairs[sim_pairs.split == "test"], ["nn_resid", "true"], ["type"])

In [ ]:
te = sim_pairs[sim_pairs.split == "test"]
plt.scatter(te["true_corr"], te["delta_nn_resid"] - te["delta"], s=2, alpha=.3)
lim = np.percentile(te["true_corr"], [1, 99]); plt.plot(lim, lim, "r")
plt.xlabel("true MV correction (Heston)"); plt.ylabel("learned correction"); plt.savefig(FIGURES_DIR / "heston_learned_vs_true.png", dpi=150)